# Retinal Blood Vessel Segmentation
**Course:** Advanced Topics in Image Analysis  
**Datasets:** DRIVE · STARE  
**Methods:** Classical Pipeline (Matched Filter + Otsu) · Random Forest Hybrid  

---
## Table of Contents
1. [Setup & Data Loading](#1-setup--data-loading)
2. [Exploratory Data Analysis](#2-exploratory-data-analysis)
3. [Pipeline Components](#3-pipeline-components)
4. [Hyperparameter Search](#4-hyperparameter-search)
5. [Final Classical Pipeline](#5-final-classical-pipeline)
6. [STARE Generalization](#6-stare-generalization)
7. [Random Forest Hybrid](#7-random-forest-hybrid)
8. [Results & Visualizations](#8-results--visualizations)

## 1. Setup & Data Loading

In [ ]:
import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score

warnings.filterwarnings('ignore')
np.random.seed(42)

In [ ]:
# ── Dataset paths (update if running locally)
DRIVE_IMAGES = '/content/DRIVE/training/images'
DRIVE_MASKS  = '/content/DRIVE/training/1st_manual'
DRIVE_FOV    = '/content/DRIVE/training/mask'

STARE_IMAGES = '/content/STARE/stare-images'
STARE_MASKS  = '/content/STARE/labels-vk'


def load_drive(img_dir, mask_dir, fov_dir):
    """Load DRIVE images, vessel masks, and FOV masks."""
    images, masks, fovs = [], [], []
    for fname in sorted(os.listdir(img_dir)):
        if fname.endswith('.tif'):
            img       = np.array(Image.open(os.path.join(img_dir,  fname)))
            mask_name = fname.replace('_training.tif', '_manual1.gif')
            fov_name  = fname.replace('.tif', '_mask.gif')
            mask      = np.array(Image.open(os.path.join(mask_dir, mask_name)))
            fov       = np.array(Image.open(os.path.join(fov_dir,  fov_name)))
            images.append(img); masks.append(mask); fovs.append(fov)
    return images, masks, fovs


def load_stare(img_dir, mask_dir):
    """Load STARE images and vessel masks."""
    images, masks = [], []
    for fname in sorted(os.listdir(img_dir)):
        if fname.endswith('.ppm') or fname.endswith('.png'):
            img       = np.array(Image.open(os.path.join(img_dir,  fname)))
            mask_name = fname.replace('.ppm', '.vk.ppm')
            mask      = np.array(Image.open(os.path.join(mask_dir, mask_name)))
            images.append(img); masks.append(mask)
    return images, masks


drive_imgs, drive_masks, drive_fovs = load_drive(DRIVE_IMAGES, DRIVE_MASKS, DRIVE_FOV)
stare_imgs, stare_masks             = load_stare(STARE_IMAGES, STARE_MASKS)

print(f'DRIVE: {len(drive_imgs)} images loaded — shape {drive_imgs[0].shape}')
print(f'STARE: {len(stare_imgs)} images loaded — shape {stare_imgs[0].shape}')

## 2. Exploratory Data Analysis
We examine dataset statistics and visualize sample images before building the pipeline.

In [ ]:
# Vessel pixel ratio — quantifies class imbalance
drive_ratios = [((m > 0).sum() / m.size * 100) for m in drive_masks]
stare_ratios = [((m > 0).sum() / m.size * 100) for m in stare_masks]

print(f'DRIVE — avg vessel pixel ratio: {np.mean(drive_ratios):.2f}%  '
      f'(min {min(drive_ratios):.2f}%, max {max(drive_ratios):.2f}%)')
print(f'STARE — avg vessel pixel ratio: {np.mean(stare_ratios):.2f}%  '
      f'(min {min(stare_ratios):.2f}%, max {max(stare_ratios):.2f}%)')

# Sample visualizations
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
fig.suptitle('DRIVE Dataset — Sample Images', fontsize=14)
for col, idx in enumerate([0, 5, 10]):
    overlay = drive_imgs[idx].copy()
    overlay[drive_masks[idx] > 0] = [255, 0, 0]
    axes[0, col].imshow(drive_imgs[idx]);       axes[0, col].set_title(f'Original #{idx+1}');    axes[0, col].axis('off')
    axes[1, col].imshow(drive_masks[idx], cmap='gray'); axes[1, col].set_title(f'Vessel Mask #{idx+1}'); axes[1, col].axis('off')
    axes[2, col].imshow(overlay);               axes[2, col].set_title(f'Overlay #{idx+1}');     axes[2, col].axis('off')
plt.tight_layout()
plt.savefig('drive_samples.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Pipeline Components
Each function below corresponds to one processing stage. They are built and tested incrementally before being composed into the full pipeline.

### 3.1 Preprocessing — Green Channel + CLAHE
Blood vessels are most visible in the green channel of retinal images. CLAHE (Contrast Limited Adaptive Histogram Equalization) enhances local contrast without over-amplifying noise.

In [ ]:
def preprocess(image, clip_limit=8.0, tile_size=24):
    """Extract green channel and apply CLAHE enhancement."""
    green = image[:, :, 1]
    clahe = cv2.createCLAHE(clipLimit=clip_limit,
                             tileGridSize=(tile_size, tile_size))
    enhanced = clahe.apply(green)
    return green, enhanced


# Visualize on the first DRIVE image
green, enhanced = preprocess(drive_imgs[0])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(axes,
                           [drive_imgs[0], green, enhanced],
                           ['Original', 'Green Channel', 'CLAHE Enhanced']):
    ax.imshow(img, cmap='gray' if img.ndim == 2 else None)
    ax.set_title(title); ax.axis('off')
plt.tight_layout()
plt.savefig('step1_enhancement.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2 Matched Filter
The matched filter is tuned to the cross-sectional Gaussian profile of blood vessels (Chaudhuri et al., 1989). A 1D Gaussian kernel is tiled into 2D and rotated across `n_angles` orientations; the maximum response at each pixel captures vessels regardless of their local direction.

In [ ]:
def matched_filter(image, sigma=4.5, filter_len=11, n_angles=12):
    """
    Apply a rotated Gaussian matched filter to detect blood vessels.

    Parameters
    ----------
    image      : uint8 grayscale image
    sigma      : std-dev of the Gaussian cross-section (controls vessel width sensitivity)
    filter_len : length of the 1D kernel (should be odd)
    n_angles   : number of rotation angles sampled in [0, 180)

    Returns
    -------
    uint8 response map, normalized to [0, 255]
    """
    half = filter_len // 2
    x = np.arange(-half, half + 1)
    kernel_1d = -np.exp(-x**2 / (2 * sigma**2))
    kernel_1d -= kernel_1d.mean()          # zero-mean → suppress uniform background
    kernel_2d = np.tile(kernel_1d, (filter_len, 1))

    responses = []
    for angle in np.linspace(0, 180, n_angles, endpoint=False):
        M = cv2.getRotationMatrix2D((filter_len // 2, filter_len // 2), angle, 1)
        rotated  = cv2.warpAffine(kernel_2d.astype(np.float32), M, (filter_len, filter_len))
        response = cv2.filter2D(image.astype(np.float32), -1, rotated)
        responses.append(response)

    result = np.max(responses, axis=0)
    result = cv2.normalize(result, None, 0, 255, cv2.NORM_MINMAX)
    return result.astype(np.uint8)


# Test on CLAHE-enhanced image
filter_response = matched_filter(enhanced)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(enhanced, cmap='gray');        axes[0].set_title('CLAHE Enhanced'); axes[0].axis('off')
axes[1].imshow(filter_response, cmap='gray'); axes[1].set_title('Matched Filter Response'); axes[1].axis('off')
plt.tight_layout()
plt.savefig('step2_matched_filter.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.3 Thresholding & FOV Masking
Otsu's method automatically finds the optimal global threshold. The FOV (Field of View) mask is then applied to exclude pixels outside the retina.

In [ ]:
def threshold_and_mask(filter_response, fov_mask):
    """Binarize with Otsu's threshold and apply FOV mask."""
    _, binary = cv2.threshold(filter_response, 0, 255,
                               cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    fov    = (fov_mask > 0).astype(np.uint8) * 255
    binary = cv2.bitwise_and(binary, fov)
    return binary


binary = threshold_and_mask(filter_response, drive_fovs[0])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(axes,
                           [filter_response, binary, drive_masks[0]],
                           ['Matched Filter', 'Otsu Binary', 'Ground Truth']):
    ax.imshow(img, cmap='gray'); ax.set_title(title); ax.axis('off')
plt.tight_layout()
plt.savefig('step3_threshold.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4 Morphological Post-processing
Closing fills small gaps in vessel contours; opening removes isolated noise blobs. Both operations use an elliptical structuring element to respect vessel curvature.

In [ ]:
def morphological_cleanup(binary, kernel_size=3, close_iter=2, open_iter=1):
    """Fill gaps (closing) and remove noise blobs (opening)."""
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    closed  = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=close_iter)
    cleaned = cv2.morphologyEx(closed,  cv2.MORPH_OPEN,  kernel, iterations=open_iter)
    return cleaned


cleaned = morphological_cleanup(binary)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(axes,
                           [binary, cleaned, drive_masks[0]],
                           ['Before Morphology', 'After Morphology', 'Ground Truth']):
    ax.imshow(img, cmap='gray'); ax.set_title(title); ax.axis('off')
plt.tight_layout()
plt.savefig('step4_morphology.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Hyperparameter Search
We perform coarse-to-fine grid search to find the best CLAHE and matched filter parameters. Each grid is evaluated using mean Dice score over all 20 DRIVE training images.

**Search strategy:**
- **Round 1 (coarse):** `clip_limit` ∈ {1, 2, 3}, `tile_size` ∈ {4, 8, 16}, `sigma` ∈ {1.5, 2.0, 2.5}, `filter_len` ∈ {7, 9, 11}
- **Round 2 (refined):** `clip_limit` ∈ {3, 4, 5}, `tile_size` ∈ {16, 24, 32}, `sigma` ∈ {2.5, 3.0, 3.5}, `filter_len` ∈ {11, 13, 15}
- **Round 3 (fine):** `clip_limit` ∈ {5, 6, 7, 8}, `tile_size` ∈ {16, 24}, `sigma` ∈ {3.5, 4.0, 4.5, 5.0}, `filter_len` ∈ {11, 13, 15}

**Optimal parameters found:** `clip_limit=8.0`, `tile_size=24`, `sigma=4.5`, `filter_len=11`

In [ ]:
def grid_search_clahe_mf(drive_imgs, drive_masks, drive_fovs,
                          clip_limits, tile_sizes, sigmas, filter_lens):
    """
    Grid search over CLAHE and matched filter hyperparameters.
    Returns results sorted by descending mean Dice score.
    """
    results = []

    for clip_limit in clip_limits:
      for tile_size in tile_sizes:
        for sigma in sigmas:
          for flen in filter_lens:

            dices = []
            for i in range(len(drive_imgs)):
                green = drive_imgs[i][:, :, 1]
                fov   = (drive_fovs[i] > 0).astype(np.uint8)

                clahe    = cv2.createCLAHE(clipLimit=clip_limit,
                                           tileGridSize=(tile_size, tile_size))
                enhanced = clahe.apply(green)
                enhanced = cv2.bitwise_and(enhanced, enhanced, mask=fov)

                fr = matched_filter(enhanced, sigma=sigma, filter_len=flen)

                _, binary = cv2.threshold(fr, 0, 255,
                                          cv2.THRESH_BINARY + cv2.THRESH_OTSU)
                binary = cv2.bitwise_and(binary, binary, mask=fov)

                kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
                closed  = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=2)
                cleaned = cv2.morphologyEx(closed,  cv2.MORPH_OPEN,  kernel, iterations=1)

                pred = (cleaned > 0).flatten()[fov.flatten() > 0]
                gt   = (drive_masks[i] > 0).flatten()[fov.flatten() > 0]
                tp   = np.sum((pred == 1) & (gt == 1))
                fp   = np.sum((pred == 1) & (gt == 0))
                fn   = np.sum((pred == 0) & (gt == 1))
                dices.append(2 * tp / (2 * tp + fp + fn + 1e-8))

            results.append((np.mean(dices), clip_limit, tile_size, sigma, flen))

    results.sort(reverse=True)
    return results


# Round 1 — coarse search
print('Round 1 — coarse search')
r1 = grid_search_clahe_mf(drive_imgs, drive_masks, drive_fovs,
                           clip_limits=[1.0, 2.0, 3.0],
                           tile_sizes=[4, 8, 16],
                           sigmas=[1.5, 2.0, 2.5],
                           filter_lens=[7, 9, 11])
print(f'Best: Dice={r1[0][0]:.4f} | clip={r1[0][1]} tile={r1[0][2]} sigma={r1[0][3]} flen={r1[0][4]}')
print('Top 5:')
for d, cl, ts, sg, fl in r1[:5]:
    print(f'  Dice={d:.4f} | clip={cl} tile={ts} sigma={sg} flen={fl}')

In [ ]:
# Round 2 — refined search around Round 1 optimum
print('Round 2 — refined search')
r2 = grid_search_clahe_mf(drive_imgs, drive_masks, drive_fovs,
                           clip_limits=[3.0, 4.0, 5.0],
                           tile_sizes=[16, 24, 32],
                           sigmas=[2.5, 3.0, 3.5],
                           filter_lens=[11, 13, 15])
print(f'Best: Dice={r2[0][0]:.4f} | clip={r2[0][1]} tile={r2[0][2]} sigma={r2[0][3]} flen={r2[0][4]}')
print('Top 5:')
for d, cl, ts, sg, fl in r2[:5]:
    print(f'  Dice={d:.4f} | clip={cl} tile={ts} sigma={sg} flen={fl}')

In [ ]:
# Round 3 — fine search
print('Round 3 — fine search')
r3 = grid_search_clahe_mf(drive_imgs, drive_masks, drive_fovs,
                           clip_limits=[5.0, 6.0, 7.0, 8.0],
                           tile_sizes=[16, 24],
                           sigmas=[3.5, 4.0, 4.5, 5.0],
                           filter_lens=[11, 13, 15])
print(f'Best: Dice={r3[0][0]:.4f} | clip={r3[0][1]} tile={r3[0][2]} sigma={r3[0][3]} flen={r3[0][4]}')
print('Top 5:')
for d, cl, ts, sg, fl in r3[:5]:
    print(f'  Dice={d:.4f} | clip={cl} tile={ts} sigma={sg} flen={fl}')

# Store best parameters for the rest of the notebook
BEST_PARAMS = {
    'clahe_clip':   8.0,
    'clahe_tile':   24,
    'gauss_sigma':  2.0,
    'mf_sigma':     4.5,
    'mf_length':    11,
    'mf_angles':    12,
    'morph_kernel': 3,
    'morph_close':  2,
    'morph_open':   1,
}
print('\nFinal best parameters:', BEST_PARAMS)

## 5. Final Classical Pipeline
The complete pipeline with the best parameters found in Section 4.

**Stages:** Green channel → CLAHE → Gaussian smoothing → Matched filter → Otsu threshold → Morphology

In [ ]:
def compute_metrics(pred_mask, gt_mask, fov_mask):
    """
    Compute segmentation metrics restricted to the FOV region.

    Returns
    -------
    dict with keys: sensitivity, specificity, accuracy, dice, f1, auc
    """
    fov  = (fov_mask > 0).flatten()
    pred = (pred_mask > 0).flatten()[fov]
    gt   = (gt_mask   > 0).flatten()[fov]

    tp = np.sum((pred == 1) & (gt == 1))
    tn = np.sum((pred == 0) & (gt == 0))
    fp = np.sum((pred == 1) & (gt == 0))
    fn = np.sum((pred == 0) & (gt == 1))

    return {
        'sensitivity': tp / (tp + fn + 1e-8),
        'specificity': tn / (tn + fp + 1e-8),
        'accuracy':    (tp + tn) / (tp + tn + fp + fn + 1e-8),
        'dice':        2 * tp / (2 * tp + fp + fn + 1e-8),
        'f1':          f1_score(gt, pred),
        'auc':         roc_auc_score(gt, pred),
    }


def full_pipeline(image, fov_mask, params=None):
    """
    Classical retinal vessel segmentation pipeline.

    Parameters
    ----------
    image    : (H, W, 3) RGB retinal image
    fov_mask : (H, W) binary mask of the retinal field of view
    params   : dict of hyperparameters (defaults to BEST_PARAMS)

    Returns
    -------
    (H, W) uint8 binary vessel mask
    """
    if params is None:
        params = BEST_PARAMS
    p = params

    # 1. Green channel
    green    = image[:, :, 1]
    fov      = (fov_mask > 0).astype(np.uint8)

    # 2. CLAHE
    clahe    = cv2.createCLAHE(clipLimit=p['clahe_clip'],
                                tileGridSize=(p['clahe_tile'], p['clahe_tile']))
    enhanced = clahe.apply(green)
    enhanced = cv2.bitwise_and(enhanced, enhanced, mask=fov)

    # 3. Gaussian smoothing
    smoothed = cv2.GaussianBlur(enhanced, (5, 5), sigmaX=p['gauss_sigma'])

    # 4. Matched filter
    fr       = matched_filter(smoothed, sigma=p['mf_sigma'],
                               filter_len=p['mf_length'], n_angles=p['mf_angles'])

    # 5. Otsu threshold
    _, binary = cv2.threshold(fr, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    binary    = cv2.bitwise_and(binary, binary, mask=fov)

    # 6. Morphological cleanup
    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                         (p['morph_kernel'], p['morph_kernel']))
    closed  = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=p['morph_close'])
    cleaned = cv2.morphologyEx(closed,  cv2.MORPH_OPEN,  kernel, iterations=p['morph_open'])

    return cleaned


# Evaluate on all 20 DRIVE training images
print('=== Classical Pipeline — DRIVE ===')
drive_classical_metrics = []
for i in range(len(drive_imgs)):
    result  = full_pipeline(drive_imgs[i], drive_fovs[i])
    metrics = compute_metrics(result, drive_masks[i], drive_fovs[i])
    drive_classical_metrics.append(metrics)
    print(f'Image {i+1:2d} — Dice: {metrics["dice"]:.4f} | '
          f'Sens: {metrics["sensitivity"]:.4f} | Spec: {metrics["specificity"]:.4f}')

print('\n--- DRIVE Averages (Classical) ---')
for key in drive_classical_metrics[0]:
    print(f'  {key:12s}: {np.mean([m[key] for m in drive_classical_metrics]):.4f}')

## 6. STARE Generalization
STARE images lack a provided FOV mask. We automatically generate one by thresholding the grayscale image and retaining the largest connected component.

In [ ]:
def generate_fov_stare(image):
    """Auto-generate a FOV mask for STARE images (no provided mask)."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    _, fov = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY)

    # Retain the largest connected component (the eye)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(fov)
    largest   = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    fov_clean = (labels == largest).astype(np.uint8) * 255

    # Smooth edges
    kernel    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    fov_clean = cv2.morphologyEx(fov_clean, cv2.MORPH_CLOSE, kernel)
    return fov_clean


# Quick visual check
fov_test = generate_fov_stare(stare_imgs[0])
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(stare_imgs[0]);          axes[0].set_title('STARE Image');   axes[0].axis('off')
axes[1].imshow(fov_test, cmap='gray'); axes[1].set_title('Auto FOV Mask'); axes[1].axis('off')
plt.tight_layout(); plt.show()


# Evaluate on all STARE images
print('=== Classical Pipeline — STARE ===')
stare_classical_metrics = []
for i in range(len(stare_imgs)):
    fov     = generate_fov_stare(stare_imgs[i])
    result  = full_pipeline(stare_imgs[i], fov)
    metrics = compute_metrics(result, stare_masks[i], fov)
    stare_classical_metrics.append(metrics)
    print(f'Image {i+1:2d} — Dice: {metrics["dice"]:.4f} | '
          f'Sens: {metrics["sensitivity"]:.4f} | Spec: {metrics["specificity"]:.4f}')

print('\n--- STARE Averages (Classical) ---')
for key in stare_classical_metrics[0]:
    print(f'  {key:12s}: {np.mean([m[key] for m in stare_classical_metrics]):.4f}')

## 7. Random Forest Hybrid
We augment the classical pipeline with a supervised Random Forest classifier trained on 7 hand-crafted pixel features. This allows the model to learn a more flexible decision boundary than the fixed Otsu threshold.

**Features per pixel:**
| # | Feature | Description |
|---|---------|-------------|
| 1 | Green | Raw green channel intensity |
| 2 | CLAHE | CLAHE-enhanced intensity |
| 3 | Gaussian | Gaussian-smoothed intensity |
| 4 | MF Large (σ=4.5) | Matched filter response — wide vessels |
| 5 | MF Small (σ=2.0) | Matched filter response — thin vessels |
| 6 | Gabor | Max Gabor response across 6 orientations |
| 7 | Gradient | Sobel gradient magnitude |

In [ ]:
def extract_features(image, fov_mask):
    """
    Extract 7 hand-crafted features per pixel.

    Returns
    -------
    features : (H, W, 7) float32 array
    fov      : (H, W) uint8 binary mask
    """
    green = image[:, :, 1]
    fov   = (fov_mask > 0).astype(np.uint8)

    # CLAHE + Gaussian
    clahe    = cv2.createCLAHE(clipLimit=8.0, tileGridSize=(24, 24))
    enhanced = clahe.apply(green)
    enhanced = cv2.bitwise_and(enhanced, enhanced, mask=fov)
    blurred  = cv2.GaussianBlur(enhanced, (5, 5), sigmaX=2.0)

    # Matched filter at two scales
    mf_large = matched_filter(blurred, sigma=4.5, filter_len=11).astype(np.float32)
    mf_small = matched_filter(blurred, sigma=2.0, filter_len=11).astype(np.float32)

    # Gabor filter (max response across 6 orientations)
    gabor_responses = [
        cv2.filter2D(blurred.astype(np.float32), -1,
                     cv2.getGaborKernel((11, 11), sigma=3.0, theta=theta,
                                        lambd=8.0, gamma=0.5, psi=0))
        for theta in np.linspace(0, np.pi, 6, endpoint=False)
    ]
    gabor = np.max(gabor_responses, axis=0)

    # Sobel gradient magnitude
    sobelx   = cv2.Sobel(blurred.astype(np.float32), cv2.CV_32F, 1, 0, ksize=3)
    sobely   = cv2.Sobel(blurred.astype(np.float32), cv2.CV_32F, 0, 1, ksize=3)
    gradient = np.sqrt(sobelx**2 + sobely**2)

    features = np.stack([
        green.astype(np.float32),
        enhanced.astype(np.float32),
        blurred.astype(np.float32),
        mf_large, mf_small, gabor, gradient
    ], axis=-1)

    return features, fov


# Build balanced training set (5000 vessel + 5000 background pixels per image)
print('Extracting features from DRIVE training images...')
X_list, y_list = [], []

for i in range(len(drive_imgs)):
    features, fov = extract_features(drive_imgs[i], drive_fovs[i])
    mask     = (drive_masks[i] > 0).astype(np.uint8)
    fov_flat = fov.flatten() > 0

    X = features.reshape(-1, 7)[fov_flat]
    y = mask.flatten()[fov_flat]

    vessel_idx = np.where(y == 1)[0]
    bg_idx     = np.where(y == 0)[0]
    n          = min(len(vessel_idx), 5000)
    idx        = np.concatenate([
        np.random.choice(vessel_idx, n, replace=False),
        np.random.choice(bg_idx,     n, replace=False)
    ])
    X_list.append(X[idx]); y_list.append(y[idx])

X_train = np.vstack(X_list)
y_train = np.concatenate(y_list)
print(f'Training set: {X_train.shape[0]:,} pixels · {X_train.shape[1]} features')

# Train Random Forest
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_scaled, y_train)
print('Random Forest training complete.')

# Feature importances
feature_names = ['Green', 'CLAHE', 'Gaussian', 'MF Large', 'MF Small', 'Gabor', 'Gradient']
print('\nFeature importances:')
for name, imp in sorted(zip(feature_names, rf.feature_importances_), key=lambda x: -x[1]):
    print(f'  {name:12s}: {imp:.4f}')

In [ ]:
def predict_rf(image, fov_mask, model, scaler):
    """Apply trained Random Forest to segment vessels."""
    features, fov = extract_features(image, fov_mask)
    H, W     = fov.shape
    fov_flat = fov.flatten() > 0

    X_fov    = features.reshape(-1, 7)[fov_flat]
    X_scaled = scaler.transform(X_fov)
    pred_fov = model.predict(X_scaled)

    pred_full = np.zeros(H * W, dtype=np.uint8)
    pred_full[fov_flat] = pred_fov
    return pred_full.reshape(H, W)


# Evaluate on DRIVE
print('=== Random Forest — DRIVE ===')
drive_rf_metrics = []
for i in range(len(drive_imgs)):
    result  = predict_rf(drive_imgs[i], drive_fovs[i], rf, scaler)
    metrics = compute_metrics(result, drive_masks[i], drive_fovs[i])
    drive_rf_metrics.append(metrics)
    print(f'Image {i+1:2d} — Dice: {metrics["dice"]:.4f} | '
          f'Sens: {metrics["sensitivity"]:.4f} | Spec: {metrics["specificity"]:.4f}')

print('\n--- DRIVE Averages (RF) ---')
for key in drive_rf_metrics[0]:
    print(f'  {key:12s}: {np.mean([m[key] for m in drive_rf_metrics]):.4f}')


# Evaluate on STARE
print('\n=== Random Forest — STARE ===')
stare_rf_metrics = []
for i in range(len(stare_imgs)):
    fov     = generate_fov_stare(stare_imgs[i])
    result  = predict_rf(stare_imgs[i], fov, rf, scaler)
    metrics = compute_metrics(result, stare_masks[i], fov)
    stare_rf_metrics.append(metrics)
    print(f'Image {i+1:2d} — Dice: {metrics["dice"]:.4f} | '
          f'Sens: {metrics["sensitivity"]:.4f} | Spec: {metrics["specificity"]:.4f}')

print('\n--- STARE Averages (RF) ---')
for key in stare_rf_metrics[0]:
    print(f'  {key:12s}: {np.mean([m[key] for m in stare_rf_metrics]):.4f}')

## 8. Results & Visualizations

In [ ]:
# Summary table
print('=' * 75)
print(f'{"Method":<30} {"Sens":>7} {"Spec":>7} {"Acc":>7} {"Dice":>7} {"AUC":>7}')
print('=' * 75)

results_table = [
    ('Classical Pipeline — DRIVE',  drive_classical_metrics),
    ('Classical Pipeline — STARE',  stare_classical_metrics),
    ('Random Forest — DRIVE',       drive_rf_metrics),
    ('Random Forest — STARE',       stare_rf_metrics),
]
for method, metrics_list in results_table:
    avgs = {k: np.mean([m[k] for m in metrics_list]) for k in metrics_list[0]}
    print(f'{method:<30} '
          f'{avgs["sensitivity"]:>7.4f} '
          f'{avgs["specificity"]:>7.4f} '
          f'{avgs["accuracy"]:>7.4f} '
          f'{avgs["dice"]:>7.4f} '
          f'{avgs["auc"]:>7.4f}')
print('=' * 75)

In [ ]:
# Pipeline visualization — all processing stages side by side
p   = BEST_PARAMS
img = drive_imgs[0]
fov = drive_fovs[0]
gt  = drive_masks[0]

green        = img[:, :, 1]
fov_mask_u8  = (fov > 0).astype(np.uint8)
clahe        = cv2.createCLAHE(clipLimit=p['clahe_clip'],
                                tileGridSize=(p['clahe_tile'], p['clahe_tile']))
enhanced     = cv2.bitwise_and(clahe.apply(green), clahe.apply(green), mask=fov_mask_u8)
smoothed     = cv2.GaussianBlur(enhanced, (5, 5), sigmaX=p['gauss_sigma'])
mf_resp      = matched_filter(smoothed, sigma=p['mf_sigma'],
                               filter_len=p['mf_length'], n_angles=p['mf_angles'])
_, binarized = cv2.threshold(mf_resp, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
binarized    = cv2.bitwise_and(binarized, binarized, mask=fov_mask_u8)
kernel       = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                          (p['morph_kernel'], p['morph_kernel']))
morph_result = cv2.morphologyEx(
    cv2.morphologyEx(binarized, cv2.MORPH_CLOSE, kernel, iterations=p['morph_close']),
    cv2.MORPH_OPEN, kernel, iterations=p['morph_open'])
rf_result    = predict_rf(img, fov, rf, scaler)

steps = [
    (img,          'Original',         False),
    (green,        'Green Channel',    True),
    (enhanced,     'CLAHE',            True),
    (smoothed,     'Gaussian',         True),
    (mf_resp,      'Matched Filter',   True),
    (binarized,    'Otsu Threshold',   True),
    (morph_result, 'Morphology',       True),
    (gt,           'Ground Truth',     True),
    (rf_result,    'RF Hybrid',        True),
]

fig, axes = plt.subplots(1, 9, figsize=(26, 4))
fig.suptitle('Retinal Vessel Segmentation Pipeline', fontsize=13, fontweight='bold')
for ax, (image, title, gray) in zip(axes, steps):
    ax.imshow(image, cmap='gray' if gray else None)
    ax.set_title(title, fontsize=8, fontweight='bold'); ax.axis('off')
plt.tight_layout()
plt.savefig('pipeline_visualization.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: pipeline_visualization.png')

In [ ]:
# Metrics comparison bar chart
methods = ['Classical\nDRIVE', 'Classical\nSTARE', 'RF Hybrid\nDRIVE', 'RF Hybrid\nSTARE']
metric_keys = ['sensitivity', 'specificity', 'dice', 'auc']
metric_labels = ['Sensitivity', 'Specificity', 'Dice', 'AUC']
all_results  = [drive_classical_metrics, stare_classical_metrics,
                drive_rf_metrics, stare_rf_metrics]

colors = ['#2196F3', '#64B5F6', '#E53935', '#EF9A9A']
x      = np.arange(len(methods))

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('Performance Metrics Comparison', fontsize=14, fontweight='bold')

for ax, key, label in zip(axes, metric_keys, metric_labels):
    values = [np.mean([m[key] for m in ml]) for ml in all_results]
    bars   = ax.bar(x, values, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_title(label, fontweight='bold', fontsize=12)
    ax.set_xticks(x); ax.set_xticklabels(methods, fontsize=9)
    ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
    ax.grid(axis='y', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('metrics_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: metrics_comparison.png')

In [ ]:
# Per-image Dice score comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Per-Image Dice Scores', fontsize=13, fontweight='bold')

x     = np.arange(20)
width = 0.35
ids   = [f'#{i+1}' for i in range(20)]

for ax, classical_m, rf_m, dataset in zip(
        axes,
        [drive_classical_metrics, stare_classical_metrics],
        [drive_rf_metrics,        stare_rf_metrics],
        ['DRIVE', 'STARE']):

    d_cl = [m['dice'] for m in classical_m]
    d_rf = [m['dice'] for m in rf_m]

    ax.bar(x - width/2, d_cl, width, label='Classical', color='#2196F3', alpha=0.85)
    ax.bar(x + width/2, d_rf, width, label='RF Hybrid', color='#E53935', alpha=0.85)
    ax.axhline(np.mean(d_cl), color='#2196F3', linestyle='--', linewidth=1.2,
               label=f'Classical avg: {np.mean(d_cl):.3f}')
    ax.axhline(np.mean(d_rf), color='#E53935', linestyle='--', linewidth=1.2,
               label=f'RF avg: {np.mean(d_rf):.3f}')
    ax.set_title(f'{dataset} Dataset', fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(ids, fontsize=7)
    ax.set_ylabel('Dice Score'); ax.set_ylim(0, 1.0)
    ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('per_image_dice.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: per_image_dice.png')

In [ ]:
# Feature importance chart
feature_names = ['Green', 'CLAHE', 'Gaussian', 'MF Large\n(σ=4.5)',
                  'MF Small\n(σ=2.0)', 'Gabor', 'Gradient']
importances   = rf.feature_importances_
sorted_idx    = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(range(7), importances[sorted_idx],
              color=['#E53935' if i == 0 else '#2196F3' for i in range(7)],
              edgecolor='white')
ax.set_xticks(range(7))
ax.set_xticklabels([feature_names[i] for i in sorted_idx], fontsize=10)
ax.set_title('Random Forest — Feature Importances', fontsize=13, fontweight='bold')
ax.set_ylabel('Importance Score')
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
for bar, val in zip(bars, importances[sorted_idx]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: feature_importance.png')